In [1]:
import os
import importlib
os.environ["CUDA_VISIBLE_DEVICES"]="1,2"
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, AutoModel
from datasets import load_dataset
import torch
#from sentence_transformers import SentenceTransformer, InputExample, losses
#from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator, SimilarityFunction
from torch.utils.data import DataLoader
from datasets import Dataset
import pandas as pd
from collections import defaultdict
import re
import numpy as np
import time

device1 = 'cuda:0'
device2 = 'cuda:1'
data_dir = '/raid/deallab/SF_RAG_Data/ASQA'
# data_dir = '../data'

/home/dataconv/anaconda3/envs/sf_rag_djk/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#load embeddings
embedd_test_path = f'{data_dir}/test/embedd_test.npy'
evidence_embeddings = np.load(embedd_test_path)
print(evidence_embeddings.shape)
evidence_embeddings = torch.from_numpy(evidence_embeddings).to(device1)

#load evidence
evidence_test_path = f'{data_dir}/test/evidence_test.csv'
evidence_df = pd.read_csv(evidence_test_path)

#load qa data
qa_df=pd.read_csv(f'{data_dir}/test/qa_test.csv') #data=df[['question','long_answers']] # questions=data['question'] #references = [row.to_dict() for i, row in df.iterrows() if i < len(questions)]
qa_df.head()

(21586, 4096)


,id,sample_id,question,follow_up_questions,long_answers,short_answers
0,c2687961-0957-45cb-bae0-42314e38f790,-7013890438520559398,Who has the highest goals in world football?,"[""Who has the highest goals in men's world int...","[""Ali Dael has the highest goals in men's worl...","[['Daei', 'Ali Daei'], ['Bican', 'Josef Bican'..."
1,26830122-8240-40a9-aaff-d9731d53b197,7089015503030534342,Who is the original artist of sound of silence?,['Who is the original artist of sound of silen...,[' The original artist of the song sound of si...,"[['Simon & Garfunkel', 'Paul Simon and Art Gar..."
2,268116a9-5ecb-4364-8da4-4a648f9d5b43,8793099883447006698,When was the first apple i phone made?,"['When was the first apple i phone released?',...",['The iPhone beta was created in 2004 to test ...,"[['June 29, 2007'], ['2004'], ['June 29, 2007...."
3,efb4810e-637b-4954-a776-3c2d05d1290c,-881464876144297194,Who played the weasley brothers in harry potter?,['Who played Bill weasley in Harry Potter and...,['Rupert Grint played Ron Weasley in all the H...,"[['Richard Fish'], ['Chris Rankin'], ['James P..."
4,99817eba-d32a-4c4d-9fe2-93a50ae1d367,1650309494326541834,How many state parks are there in virginia?,['How many state parks are there in virginia i...,['When the Virginia state park system was form...,"[['six'], ['38'], ['6'], ['38']]"


In [3]:
#load quantized model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_storage=torch.bfloat16,
)

# load model with tokenizer
model = AutoModel.from_pretrained(
    'nvidia/NV-Embed-v2', 
    trust_remote_code=True,
    quantization_config = bnb_config,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage =True,
)
model.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.11s/it]


NVEmbedModel(
  (latent_attention_model): LatentAttentionModel(
    (cross_attend_blocks): ModuleList(
      (0): PreNorm(
        (fn): Attention(
          (to_q): Linear4bit(in_features=4096, out_features=32768, bias=False)
          (to_kv): Linear4bit(in_features=4096, out_features=65536, bias=False)
          (to_out): Linear4bit(in_features=32768, out_features=4096, bias=False)
        )
        (norm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
        (norm_context): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
      )
      (1): PreNorm(
        (fn): FeedForward(
          (net): Sequential(
            (0): Linear4bit(in_features=4096, out_features=32768, bias=True)
            (1): GEGLU()
            (2): Linear4bit(in_features=16384, out_features=4096, bias=True)
          )
        )
        (norm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
      )
    )
  )
  (embedding_model): BidirectionalMistralModel(
    (embed_tokens): Embedding(

In [4]:
#load tokenizer
tokenizer_gen = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")
tokenizer_gen.pad_token = tokenizer_gen.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    # bnb_4bit_quant_type="nf4",
    # bnb_4bit_compute_dtype=torch.bfloat16,
    # bnb_4bit_use_double_quant=True,
    # bnb_4bit_quant_storage=torch.bfloat16,
)

model_gen = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map= 'auto'
)
model_gen.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.44it/s]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps

In [9]:
# retrive docs from the document embeddings
def retrieve_documents(query):
    max_length = 1024
    
    #query prefix
    task_name_to_instruct = {"example": "Given a question, retrieve passages that answer the question",}
    query_prefix = "Instruct: "+task_name_to_instruct["example"]+"\nQuery: "
    
    query_embedding = model.encode([query],instruction=query_prefix, max_length=max_length).to(device1)

    similarities = torch.nn.functional.cosine_similarity(query_embedding, evidence_embeddings)

    top_results = similarities.argsort(descending=True)[:10].cpu().detach().numpy()
    res=[evidence_df.loc[idx, 'text'] for idx in top_results if idx < len(evidence_df)]
        
    return res

In [10]:
def summarize(query, docs):
    prompt = """
    In a Retrieval Augmentation Generation system, documents close to the query vector are as follows:
    ---------------------
    {0}
    ---------------------
    Identify entities and contexts in a query, and use them to extract and summarize only relevant content from documents.
    Query: {1}
    Answer:
    """.format('\n'.join(docs), query)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device2)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device2)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return [re.sub('\n|<\|eot_id\|>', '', res)]

In [11]:
def answer(query, context):
    prompt = """
    Context information is below.
    ---------------------
    {0}
    ---------------------
    Given the context information and not prior knowledge, answer the query.
    Query: {1}
    Answer:
    """.format('\n'.join(context), query)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device1)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device1)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return [re.sub('\n|<\|eot_id\|>', '', res)]

In [12]:
from tqdm import tqdm
from evaluation import evaluate

stop_iteration = 100

scores_list=[]
for idx, row in tqdm(qa_df.iterrows(), total=min([stop_iteration, len(qa_df)])):
    if idx == stop_iteration: break
    query = row['question']
    retrieved_docs = retrieve_documents(query)
    summary=summarize(query,retrieved_docs)
    print(summary)
    ans=answer(query,summary)
    scores=evaluate(ans, [row.to_dict()])
    scores_list.append(scores)
    scores_df=pd.DataFrame(scores_list)
    print(scores_df.mean())
        
scores_df=pd.DataFrame(scores_list)
scores_df.mean()

  0%|          | 0/100 [00:00<?, ?it/s]

['Based on the provided documents, the answer to the query "Who has the highest goals in world football?" is Ali Daei of Iran with 109 international goals. To answer the query, we need to identify the entity "Ali Daei" and the context "highest goals in world football". We can then extract the relevant content from the document "List of top international men\'s association football goal scorers by.../List of players" which ranks the top goal scorers in international football. The document provides a list of the top goal scorers, and Ali Daei is ranked #1 with 109 international goals.']
Based on the provided context information, the answer to the query "Who has the highest goals in world football?" is Ali Daei of Iran with 109 international goals.
Who has the highest goals in world football?
["Who has the highest goals in men's world international football?", "Who has the highest goals all-time in men's football?", "Who has the highest goals in women's world international football?"]
[['

  1%|          | 1/100 [00:10<16:30, 10.01s/it]

follow question : Who has the highest goals in men's world international football?
short answer : ['Daei', 'Ali Daei']
{'score': 0.5527681708335876, 'start': 117, 'end': 125, 'answer': 'Ali Daei'}
follow question : Who has the highest goals all-time in men's football?
short answer : ['Bican', 'Josef Bican']
{'score': 0.0009265830740332603, 'start': 117, 'end': 125, 'answer': 'Ali Daei'}
follow question : Who has the highest goals in women's world international football?
short answer : ['Sinclair', 'Christine Sinclair']
{'score': 0.0937177911400795, 'start': 117, 'end': 125, 'answer': 'Ali Daei'}
rougeLsum      38.235294
length         28.000000
str_em         33.333333
Disambig-F1    33.333333
dtype: float64
['The original artist of "The Sound of Silence" is Simon & Garfunkel, a music duo composed of Paul Simon and Art Garfunkel.']
The original artist of "The Sound of Silence" is Simon & Garfunkel.
Who is the original artist of sound of silence?
['Who is the original artist of sound of

  2%|▏         | 2/100 [00:15<11:52,  7.27s/it]

...

follow question : Who does Charlie Day play on It's Always Sunny in Philadelphia?
short answer : ['Charlie Kelly']
{'score': 0.9443321228027344, 'start': 18, 'end': 25, 'answer': 'Charlie'}
follow question : Who plays Charlie Kelly on It's Always Sunny in Philadelphia?
short answer : ['Charlie Day']
{'score': 0.2701495289802551, 'start': 0, 'end': 11, 'answer': 'Charlie Day'}
rougeLsum      24.990201
length         28.000000
str_em         34.375000
Disambig-F1    31.041667
dtype: float64
['Based on the documents provided, the answer to the query is:The Los Angeles Lakers have won the NBA Finals 16 times.']
The Los Angeles Lakers have won the NBA Finals 17 times.
How many times have the lakers won the finals?
['As of 2017, how many times have the lakers won the finals?', 'As of 2016, how many times have the Lakers won the finals?', 'As of 2015, how many times have the Lakers won the finals?']
[['16'], ['16'], ['16']]


  9%|▉         | 9/100 [01:35<13:27,  8.87s/it]

follow question : As of 2017, how many times have the lakers won the finals?
short answer : ['16']
{'score': 0.8372417092323303, 'start': 47, 'end': 49, 'answer': '17'}
follow question : As of 2016, how many times have the Lakers won the finals?
short answer : ['16']
{'score': 0.8053435683250427, 'start': 47, 'end': 49, 'answer': '17'}
follow question : As of 2015, how many times have the Lakers won the finals?
short answer : ['16']
{'score': 0.7963420748710632, 'start': 47, 'end': 49, 'answer': '17'}
rougeLsum      24.637754
length         26.111111
str_em         30.555556
Disambig-F1    27.592593
dtype: float64


  9%|▉         | 9/100 [01:35<16:10, 10.66s/it]


KeyboardInterrupt: 

In [ ]:
import spacy
# from transformers import pipeline

# ------------------------------------------------
# 1. 초기 설정 및 엔티티 추출 함수
# ------------------------------------------------

# spaCy 모델 로드 (한국어 문서라면 적절한 모델로 교체)
nlp = spacy.load("en_core_web_sm")

def extract_entities_with_spacy(text):
    """
    입력 텍스트에서 spaCy를 통해 엔티티를 추출하고, 
    중복을 제거한 엔티티 리스트를 반환합니다.
    """
    doc = nlp(text)
    return list({ent.text for ent in doc.ents})

# ------------------------------------------------
# 2. Llama-3.1 기반 텍스트 생성 모델 설정
# ------------------------------------------------

# Llama-3.1 기반 텍스트 생성 파이프라인 (모델 및 토크나이저 경로를 실제 환경에 맞게 수정)
# generation_model = pipeline(
#     "text-generation",
#     model="path/to/llama-3.1-generation-model",    # 실제 모델 경로로 변경
#     tokenizer="path/to/llama-3.1-generation-model"   # 실제 토크나이저 경로로 변경
# )

# ------------------------------------------------
# 3. 문서 검색 및 답변 생성을 위한 함수들
# ------------------------------------------------

# 예시 문서 리스트 (실제 시스템에서는 다수의 문서를 사용)
documents = [
    "홍길동은 서울대학교에서 연구를 수행한다.",
    "김영희와 이철수는 공동 연구 프로젝트를 진행한다.",
    "이영수는 현대자동차에서 개발을 이끌고 있다."
]

def retrieve_documents(query, documents):
    """
    모호한 질의(query)에 대해, 간단한 키워드 매칭으로 관련 문서를 검색합니다.
    만약 매칭되는 문서가 없으면, 모든 문서를 반환합니다.
    """
    query_words = query.lower().split()
    relevant_docs = []
    for doc in documents:
        doc_lower = doc.lower()
        if any(qw in doc_lower for qw in query_words):
            relevant_docs.append(doc)
    if not relevant_docs:
        relevant_docs = documents  # 매칭되는 문서가 없으면 전체 문서를 대상으로 함
    return relevant_docs

def generate_answer_for_document(doc, query):
    """
    단일 문서(doc)에 대해, spaCy를 통해 추출한 개체 중 질의(query)와 관련된 
    (간단한 키워드 매칭 기반) 개체들을 중심으로 해당 문서의 답변(요약)을 생성합니다.
    """
    # ① 문서 내 엔티티 추출
    entities = extract_entities_with_spacy(doc)
    
    # ② 질의와 관련된 엔티티 필터링 (질의에 포함된 단어가 엔티티에 존재하면 선택)
    query_words = query.lower().split()
    relevant_entities = [entity for entity in entities if any(qw in entity.lower() for qw in query_words)]
    
    # 관련 엔티티가 없는 경우 전체 엔티티 사용
    if not relevant_entities:
        relevant_entities = entities
    
    # ③ 생성 모델 프롬프트 구성
    prompt = (
        f"다음 문서를 읽고, 모호한 질의 '{query}'와 관련된 개체들(예: {', '.join(relevant_entities) if relevant_entities else '해당 없음'})를 중심으로 상세한 답변을 작성하라.\n"
        f"문서: {doc}\n"
        f"답변:"
    )
    
    # ④ 답변(요약) 생성
    answer_snippet = model_gen(prompt, max_length=256)[0]['generated_text']
    return answer_snippet

def combine_document_answers(answer_list, query):
    """
    여러 문서에서 생성된 답변(answer_list)을 하나의 일관된 장문 답변으로 통합합니다.
    """
    combined_context = "\n\n".join([f"문서 {i+1}에 대한 답변: {ans}" for i, ans in enumerate(answer_list)])
    prompt = (
        f"다음은 모호한 질의 '{query}'에 대해 여러 문서에서 생성된 답변들이다. "
        f"이들을 종합하여 하나의 일관된 장문 답변을 작성하라.\n\n"
        f"{combined_context}\n\n"
        f"최종 답변:"
    )
    final_answer = model_gen(prompt, max_length=512)[0]['generated_text']
    return final_answer

def generate_final_answer(query, documents):
    """
    1. 모호한 질의에 대해 관련 문서를 검색하고,
    2. 각 문서별로 질의와 관련된 개체를 활용하여 답변을 생성한 후,
    3. 생성된 답변들을 하나의 장문 답변으로 통합하여 반환합니다.
    """
    # ① 질의에 맞는 문서 검색
    retrieved_docs = retrieve_documents(query, documents)
    
    # ② 각 문서에서 답변 생성
    answer_snippets = []
    for doc in retrieved_docs:
        snippet = generate_answer_for_document(doc, query)
        answer_snippets.append(snippet)
    
    # ③ 개별 답변들을 통합하여 최종 답변 생성
    final_answer = combine_document_answers(answer_snippets, query)
    return final_answer

# ------------------------------------------------
# 4. 모호한 질의에 대해 최종 답변 생성 실행
# ------------------------------------------------

ambiguous_query = "연구 프로젝트 정보"
final_answer = generate_final_answer(ambiguous_query, documents)

print("=== 최종 생성 답변 ===")
print(final_answer)
